In [1]:
import pandas as pd

# Load smaller sample data

orders = pd.read_csv(
    '../data/orders.csv'
)

order_products = pd.read_csv(
    '../data/order_products__prior.csv',
    nrows=300000
)

# Keep only orders with at least 2 products
order_counts = order_products['order_id'].value_counts()

valid_orders = order_counts[order_counts >= 2].index

order_products = order_products[
    order_products['order_id'].isin(valid_orders)
]

products = pd.read_csv(
    '../data/products.csv'
)

print("Orders:", orders.shape)
print("Order Products:", order_products.shape)

# Merge datasets

df = pd.merge(
    order_products,
    products,
    on='product_id'
)
print("After Merge:",df.shape)
print(df.head())
df = pd.merge(
    df,
    orders[['order_id']],
    on='order_id'
)

# Rename columns

df = df.rename(columns={
    'order_id': 'transaction_id',
    'product_id': 'item_id',
    'product_name': 'item_name'
})

# Keep required columns only

df = df[
    [
        'transaction_id',
        'item_name'
    ]
]

# Keep only most frequent products

top_products = (
    df['item_name']
    .value_counts()
    .head(100)
    .index
)

df = df[
    df['item_name'].isin(top_products)
]

print("Filtered Dataset Shape:", df.shape)

print(df.head())
print("Unique Transactions:", df['transaction_id'].nunique())
print("Unique Products:", df['item_name'].nunique())
print(df['item_name'].value_counts().head(20))

Orders: (3421083, 7)
Order Products: (298518, 4)
After Merge: (298518, 7)
   order_id  product_id  add_to_cart_order  reordered           product_name  \
0         2       33120                  1          1     Organic Egg Whites   
1         2       28985                  2          1  Michigan Organic Kale   
2         2        9327                  3          0          Garlic Powder   
3         2       45918                  4          1         Coconut Butter   
4         2       30035                  5          0      Natural Sweetener   

   aisle_id  department_id  
0        86             16  
1        83              4  
2       104             13  
3        19             13  
4        17             13  
Filtered Dataset Shape: (68713, 2)
    transaction_id               item_name
1                2   Michigan Organic Kale
5                2                 Carrots
10               3  Unsweetened Almondmilk
12               3    Organic Baby Spinach
14               3   

In [2]:
# Create lightweight transaction matrix

transaction_matrix = pd.crosstab(
    df['transaction_id'],
    df['item_name']
)

# Convert to boolean

transaction_matrix = transaction_matrix > 0

print("Transaction Matrix Shape:", transaction_matrix.shape)

transaction_matrix.head()
print(transaction_matrix.shape)
print(transaction_matrix.columns[:20])

Transaction Matrix Shape: (21306, 100)
(21306, 100)
Index(['100% Raw Coconut Water', '100% Whole Wheat Bread',
       '2% Reduced Fat Milk', 'Apple Honeycrisp Organic', 'Asparagus',
       'Bag of Organic Bananas', 'Banana', 'Bartlett Pears', 'Blueberries',
       'Boneless Skinless Chicken Breasts', 'Broccoli Crown',
       'Bunched Cilantro', 'Carrots', 'Clementines, Bag', 'Cucumber Kirby',
       'Extra Virgin Olive Oil', 'Fresh Cauliflower', 'Garlic',
       'Granny Smith Apples', 'Grape White/Green Seedless'],
      dtype='str', name='item_name')


In [3]:
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

# Generate frequent itemsets

frequent_itemsets = apriori(
    transaction_matrix,
    min_support=0.002,
    use_colnames=True,
    low_memory=True
)

print("Frequent Itemsets Shape:", frequent_itemsets.shape)

print(frequent_itemsets.head(20))
print(frequent_itemsets.shape)
print(frequent_itemsets.head(20))
print(frequent_itemsets['itemsets'].tail(20))

Frequent Itemsets Shape: (1021, 2)
     support                                        itemsets
0   0.016240             frozenset({100% Raw Coconut Water})
1   0.026518             frozenset({100% Whole Wheat Bread})
2   0.017507                frozenset({2% Reduced Fat Milk})
3   0.037032           frozenset({Apple Honeycrisp Organic})
4   0.027598                          frozenset({Asparagus})
5   0.166103             frozenset({Bag of Organic Bananas})
6   0.207078                             frozenset({Banana})
7   0.017366                     frozenset({Bartlett Pears})
8   0.025814                        frozenset({Blueberries})
9   0.022951  frozenset({Boneless Skinless Chicken Breasts})
10  0.016850                     frozenset({Broccoli Crown})
11  0.019807                   frozenset({Bunched Cilantro})
12  0.031634                            frozenset({Carrots})
13  0.016944                   frozenset({Clementines, Bag})
14  0.043368                     frozenset({Cucumb

In [4]:
rules_df = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.05
)

print(rules_df.head())
print(rules_df.shape)

                           antecedents                          consequents  \
0  frozenset({100% Raw Coconut Water})  frozenset({Bag of Organic Bananas})   
1  frozenset({100% Raw Coconut Water})                  frozenset({Banana})   
2  frozenset({100% Raw Coconut Water})    frozenset({Organic Baby Spinach})   
3  frozenset({100% Raw Coconut Water})    frozenset({Organic Hass Avocado})   
4  frozenset({100% Raw Coconut Water})    frozenset({Organic Strawberries})   

   antecedent support  consequent support   support  confidence      lift  \
0             0.01624            0.166103  0.003473    0.213873  1.287588   
1             0.01624            0.207078  0.002018    0.124277  0.600149   
2             0.01624            0.104525  0.002394    0.147399  1.410184   
3             0.01624            0.093072  0.002675    0.164740  1.770019   
4             0.01624            0.110908  0.002018    0.124277  1.120548   

   representativity  leverage  conviction  zhangs_metric   jac

In [5]:
rules_df = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.01
)
print("Frequent itemsets:", frequent_itemsets.shape)
print("Rules:", rules_df.shape)
print(frequent_itemsets[['itemsets', 'support']].head(10))
print("Rules Shape:", rules_df.shape)

print(rules_df.head())

Frequent itemsets: (1021, 2)
Rules: (2114, 14)
                                         itemsets   support
0             frozenset({100% Raw Coconut Water})  0.016240
1             frozenset({100% Whole Wheat Bread})  0.026518
2                frozenset({2% Reduced Fat Milk})  0.017507
3           frozenset({Apple Honeycrisp Organic})  0.037032
4                          frozenset({Asparagus})  0.027598
5             frozenset({Bag of Organic Bananas})  0.166103
6                             frozenset({Banana})  0.207078
7                     frozenset({Bartlett Pears})  0.017366
8                        frozenset({Blueberries})  0.025814
9  frozenset({Boneless Skinless Chicken Breasts})  0.022951
Rules Shape: (2114, 14)
                           antecedents                          consequents  \
0  frozenset({Bag of Organic Bananas})  frozenset({100% Raw Coconut Water})   
1  frozenset({100% Raw Coconut Water})  frozenset({Bag of Organic Bananas})   
2  frozenset({100% Raw Coconut W

In [6]:
# Convert antecedents and consequents into readable text

rules_df['antecedents'] = rules_df['antecedents'].apply(
    lambda items: ', '.join([str(item) for item in items])
)

rules_df['consequents'] = rules_df['consequents'].apply(
    lambda items: ', '.join([str(item) for item in items])
)

print(rules_df[['antecedents', 'consequents']].head())

              antecedents             consequents
0  Bag of Organic Bananas  100% Raw Coconut Water
1  100% Raw Coconut Water  Bag of Organic Bananas
2  100% Raw Coconut Water                  Banana
3  100% Raw Coconut Water    Organic Baby Spinach
4    Organic Baby Spinach  100% Raw Coconut Water


In [7]:
# Save rules dataset
print("Frequent itemsets:", frequent_itemsets.shape)
print(frequent_itemsets[['itemsets', 'support']].head(20))

print("Rules:", rules_df.shape)
rules_df.to_csv('../data/association_rules.csv', index=False)

print("association_rules.csv file created successfully")

Frequent itemsets: (1021, 2)
                                          itemsets   support
0              frozenset({100% Raw Coconut Water})  0.016240
1              frozenset({100% Whole Wheat Bread})  0.026518
2                 frozenset({2% Reduced Fat Milk})  0.017507
3            frozenset({Apple Honeycrisp Organic})  0.037032
4                           frozenset({Asparagus})  0.027598
5              frozenset({Bag of Organic Bananas})  0.166103
6                              frozenset({Banana})  0.207078
7                      frozenset({Bartlett Pears})  0.017366
8                         frozenset({Blueberries})  0.025814
9   frozenset({Boneless Skinless Chicken Breasts})  0.022951
10                     frozenset({Broccoli Crown})  0.016850
11                   frozenset({Bunched Cilantro})  0.019807
12                            frozenset({Carrots})  0.031634
13                   frozenset({Clementines, Bag})  0.016944
14                     frozenset({Cucumber Kirby})  0.04

In [8]:
# Testing recommendation products

for product in rules_df['antecedents'].head(10):

    print("Product:", product)

    matched_rules = rules_df[
        rules_df['antecedents'].str.contains(product, case=False)
    ]

    print(
        matched_rules[['antecedents', 'consequents']]
        .head(3)
    )

    print("-" * 50)

Product: Bag of Organic Bananas
               antecedents               consequents
0   Bag of Organic Bananas    100% Raw Coconut Water
9   Bag of Organic Bananas    100% Whole Wheat Bread
23  Bag of Organic Bananas  Apple Honeycrisp Organic
--------------------------------------------------
Product: 100% Raw Coconut Water
              antecedents             consequents
1  100% Raw Coconut Water  Bag of Organic Bananas
2  100% Raw Coconut Water                  Banana
3  100% Raw Coconut Water    Organic Baby Spinach
--------------------------------------------------
Product: 100% Raw Coconut Water
              antecedents             consequents
1  100% Raw Coconut Water  Bag of Organic Bananas
2  100% Raw Coconut Water                  Banana
3  100% Raw Coconut Water    Organic Baby Spinach
--------------------------------------------------
Product: 100% Raw Coconut Water
              antecedents             consequents
1  100% Raw Coconut Water  Bag of Organic Bananas
2  100%

In [9]:
# Products having recommendations

recommended_products = rules_df['antecedents'].unique()

print("Products with recommendations:\n")

count = 1

for itemset in recommended_products[:50]:

    # Convert frozenset to list
    products = list(itemset)

    # Convert list to readable text
    print(f"{count}. {', '.join(products)}")

    count += 1

Products with recommendations:

1. B, a, g,  , o, f,  , O, r, g, a, n, i, c,  , B, a, n, a, n, a, s
2. 1, 0, 0, %,  , R, a, w,  , C, o, c, o, n, u, t,  , W, a, t, e, r
3. O, r, g, a, n, i, c,  , B, a, b, y,  , S, p, i, n, a, c, h
4. O, r, g, a, n, i, c,  , H, a, s, s,  , A, v, o, c, a, d, o
5. O, r, g, a, n, i, c,  , S, t, r, a, w, b, e, r, r, i, e, s
6. 1, 0, 0, %,  , W, h, o, l, e,  , W, h, e, a, t,  , B, r, e, a, d
7. B, a, n, a, n, a
8. O, r, g, a, n, i, c,  , A, v, o, c, a, d, o
9. 2, %,  , R, e, d, u, c, e, d,  , F, a, t,  , M, i, l, k
10. A, p, p, l, e,  , H, o, n, e, y, c, r, i, s, p,  , O, r, g, a, n, i, c
11. C, a, r, r, o, t, s
12. L, a, r, g, e,  , L, e, m, o, n
13. L, i, m, e, s
14. O, r, g, a, n, i, c,  , C, u, c, u, m, b, e, r
15. O, r, g, a, n, i, c,  , G, a, r, l, i, c
16. O, r, g, a, n, i, c,  , G, a, r, n, e, t,  , S, w, e, e, t,  , P, o, t, a, t, o,  , (, Y, a, m, )
17. O, r, g, a, n, i, c,  , G, r, a, n, n, y,  , S, m, i, t, h,  , A, p, p, l, e
18. O, r, g, a, n, i